# Olist — exploratory analysis

This notebook explores the modelled layers built by `src/build.py`. It is the
*exploration* surface: the numbers that survive scrutiny here are promoted into
`sql/40_kpi/`, exported by `src/export_bi.py`, and only then charted.

Nothing here queries the raw CSVs directly. Everything reads `mart.*`, which is
one-row-per-order by construction, so no cell in this notebook can accidentally
multiply an amount by joining a fan-out table.

**Reproduce**: `python src/ingest.py --download && python src/build.py`, then run
this notebook top to bottom.

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from config import DB_PATH

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

con = duckdb.connect(str(DB_PATH), read_only=True)
q = lambda sql: con.execute(sql).df()

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)
print("database:", DB_PATH.name, f"({DB_PATH.stat().st_size / 1_048_576:.0f} MB)")

duckdb 1.1.3 | pandas 2.3.3
database: olist.duckdb (137 MB)


## 1. What is in the model

Five schemas, each with one job: `raw` (untouched text), `stg` (typed),
`qa` (quality checks), `int` (collapsed to order grain), `mart` (facts and KPIs).

In [2]:
q('''
SELECT table_schema, count(*) AS n_relations
FROM information_schema.tables
WHERE table_schema IN ('raw','stg','qa','int','mart')
GROUP BY 1 ORDER BY 1
''')

,table_schema,n_relations
0,int,3
1,mart,11
2,qa,4
3,raw,9
4,stg,9


## 2. The grain claim, checked here too

`fct_orders` claims one row per order. A notebook that trusts that claim without
testing it is how a wrong number reaches a slide.

In [3]:
q('''
SELECT
  count(*)                    AS rows_in_fct_orders,
  count(DISTINCT order_id)    AS distinct_orders,
  count(*) - count(DISTINCT order_id) AS duplicates
FROM mart.fct_orders
''')

,rows_in_fct_orders,distinct_orders,duplicates
0,99441,99441,0


### Money is conserved

The same total must appear at every layer. If it grows, a join multiplied rows;
if it shrinks, a join dropped them.

In [4]:
q('''
SELECT 'stg.order_items'      AS layer, sum(item_price)  AS items_value FROM stg.order_items
UNION ALL SELECT 'int.items_by_order',  sum(items_value) FROM int.items_by_order
UNION ALL SELECT 'mart.fct_orders',     sum(items_value) FROM mart.fct_orders
UNION ALL SELECT 'mart.fct_order_items',sum(item_price)  FROM mart.fct_order_items
''')

,layer,items_value
0,stg.order_items,13591643.7
1,int.items_by_order,13591643.7
2,mart.fct_orders,13591643.7
3,mart.fct_order_items,13591643.7


### Three different measures, three different numbers

Item value, shipping and cash collected are not interchangeable, and none of
them is profit — this dataset has no cost, tax, refund or commission data.

In [5]:
q('''
SELECT
  sum(items_value)        AS items_value,
  sum(freight_value)      AS freight_value,
  sum(items_plus_freight) AS items_plus_freight,
  sum(payments_value)     AS payments_value,
  sum(payments_value) - sum(items_plus_freight) AS payments_minus_items_freight
FROM mart.fct_orders
''')

,items_value,freight_value,items_plus_freight,payments_value,payments_minus_items_freight
0,13591643.7,2251909.54,15843553.24,16008872.12,165318.88


## 3. Q1 — how does item value move over time?

Boundary months are excluded (`in_analysis_window`): 2016-09 holds 4 orders and
2016-12 holds exactly 1. Leaving them in puts a cliff at each end of the series
that is an artefact of coverage, not a business event.

In [6]:
q('''
SELECT
  CASE WHEN in_analysis_window THEN 'inside 2017-01..2018-08' ELSE 'boundary months' END AS window,
  count(*)        AS n_orders,
  min(purchase_month) AS first_month,
  max(purchase_month) AS last_month
FROM mart.fct_orders
WHERE is_sale_eligible
GROUP BY 1 ORDER BY 1
''')

,window,n_orders,first_month,last_month
0,boundary months,297,2016-09-01,2018-09-01
1,inside 2017-01..2018-08,97910,2017-01-01,2018-08-01


In [7]:
sales = q("SELECT * FROM mart.kpi_sales_monthly")
sales[["purchase_month","n_orders","items_value","aov","avg_item_price","freight_pct_of_items"]]

,purchase_month,n_orders,items_value,aov,avg_item_price,freight_pct_of_items
0,2017-01-01,787,120098.27,152.60,126.02,14.03
1,2017-02-01,1718,244959.35,142.58,126.53,15.76
2,2017-03-01,2617,368341.32,140.75,123.81,15.55
3,2017-04-01,2377,353842.98,148.86,133.02,14.70
4,2017-05-01,3640,503159.19,138.23,122.54,15.81
5,2017-06-01,3205,429916.61,134.14,120.39,16.22
6,2017-07-01,3946,492287.30,124.76,109.57,17.56
7,2017-08-01,4272,568245.79,133.02,116.40,16.48
8,2017-09-01,4227,621415.91,147.01,129.06,15.40
9,2017-10-01,4547,660179.62,145.19,124.56,15.84


**Reading it.** Volume grew roughly eightfold. But average order value *fell*
and shipping's share of item value *rose* — growth is coming from more, smaller
orders, and freight is taking a larger slice. A revenue line alone hides both.

In [8]:
first, last = sales.iloc[0], sales.iloc[-1]
pd.DataFrame({
    "metric": ["orders", "items_value", "aov", "avg_item_price", "freight_pct_of_items"],
    "2017-01": [first.n_orders, first.items_value, first.aov, first.avg_item_price, first.freight_pct_of_items],
    "2018-08": [last.n_orders, last.items_value, last.aov, last.avg_item_price, last.freight_pct_of_items],
})

,metric,2017-01,2018-08
0,orders,787.00,6421.00
1,items_value,120098.27,848860.10
2,aov,152.60,132.20
3,avg_item_price,126.02,117.64
4,freight_pct_of_items,14.03,17.45


## 4. Q2 — where do delays concentrate?

First the denominator. A late rate is meaningless without saying what it is a
share *of*, so start by counting what can even be measured.

In [9]:
q('''
SELECT
  count(*)                                       AS all_orders,
  count(*) FILTER (WHERE is_sale_eligible)       AS sale_eligible,
  count(*) FILTER (WHERE order_status='delivered') AS delivered,
  count(*) FILTER (WHERE has_delivery_measurement) AS measurable,
  count(*) FILTER (WHERE order_status='delivered' AND NOT has_delivery_measurement) AS delivered_but_unmeasurable
FROM mart.fct_orders
''')

,all_orders,sale_eligible,delivered,measurable,delivered_but_unmeasurable
0,99441,98207,96478,96470,8


The typical order is not late — it is *early*, by a wide margin. Lateness is
a tail, which is why the median is reported next to the rate.

In [10]:
q('''
SELECT
  count(*)                          AS n_measurable,
  count(*) FILTER (WHERE is_late)   AS n_late,
  round(100.0*count(*) FILTER (WHERE is_late)/count(*), 2) AS late_rate_pct,
  median(delay_days)                AS median_delay_days,
  quantile_cont(delay_days, 0.9)    AS p90_delay_days,
  quantile_cont(delay_days, 0.99)   AS p99_delay_days,
  median(delivery_days)             AS median_delivery_days
FROM mart.fct_orders
WHERE has_delivery_measurement AND in_analysis_window
''')

,n_measurable,n_late,late_rate_pct,median_delay_days,p90_delay_days,p99_delay_days,median_delivery_days
0,96203,6531,6.79,-12.0,-2.0,18.0,10.0


### By category — flat

Note the counting rule: an order is counted once per category it contains, so
these order counts sum to more than the order total. That is correct for a
multi-category basket, and is stated rather than papered over.

In [11]:
q('''
SELECT category, n_orders, n_late_orders, late_rate_pct, rate_gap_pp, excess_late_orders
FROM mart.kpi_category_priority
ORDER BY n_orders DESC LIMIT 10
''')

,category,n_orders,n_late_orders,late_rate_pct,rate_gap_pp,excess_late_orders
0,bed_bath_table,9267,689,7.43,0.64,60.0
1,health_beauty,8610,647,7.51,0.72,62.0
2,sports_leisure,7512,495,6.59,-0.20,-15.0
3,computers_accessories,6517,416,6.38,-0.41,-26.0
4,furniture_decor,6258,449,7.17,0.38,24.0
5,housewares,5734,308,5.37,-1.42,-81.0
6,watches_gifts,5489,406,7.40,0.61,33.0
7,telephony,4086,291,7.12,0.33,14.0
8,auto,3801,278,7.31,0.52,20.0
9,toys,3781,243,6.43,-0.36,-14.0


### By destination state — not flat at all

Category gaps are under 1.5 points. State gaps reach 14 points. The delay
problem is geographic.

In [12]:
q('''
SELECT customer_state, n_orders, n_late_orders, late_rate_pct, rate_gap_pp,
       excess_late_orders, median_delivery_days, median_promised_days, median_handover_days
FROM mart.kpi_state_priority LIMIT 12
''')

,customer_state,n_orders,n_late_orders,late_rate_pct,rate_gap_pp,excess_late_orders,median_delivery_days,median_promised_days,median_handover_days
0,RJ,12310,1495,12.14,5.35,659.0,12.0,25.0,2.0
1,BA,3253,396,12.17,5.38,175.0,17.0,30.0,2.0
2,CE,1273,176,13.83,7.04,90.0,18.0,32.0,2.0
3,ES,1992,214,10.74,3.95,79.0,14.0,26.0,2.0
4,MA,713,125,17.53,10.74,77.0,19.0,31.0,2.0
5,AL,396,85,21.46,14.67,58.0,22.0,32.0,3.0
6,SC,3537,291,8.23,1.44,51.0,13.0,26.0,2.0
7,PE,1587,153,9.64,2.85,45.0,16.0,32.0,2.0
8,PA,942,106,11.25,4.46,42.0,21.0,37.0,2.5
9,PI,475,66,13.89,7.10,34.0,16.0,30.0,2.0


**Where in the journey?** Handover to the carrier takes about two days
everywhere, in fast states and slow ones alike. The variation is downstream of
that, in transit and final delivery — which points at carrier and route, not at
seller dispatch speed.

In [13]:
q('''
SELECT
  CASE WHEN late_rate_pct >= 12 THEN 'states at or above 12%' ELSE 'states below 12%' END AS band,
  count(*)                     AS n_states,
  sum(n_orders)                AS n_orders,
  round(avg(median_handover_days), 2)  AS avg_median_handover_days,
  round(avg(median_delivery_days), 2)  AS avg_median_delivery_days,
  round(avg(median_promised_days), 2)  AS avg_median_promised_days
FROM mart.kpi_state_priority
GROUP BY 1 ORDER BY 1
''')

,band,n_states,n_orders,avg_median_handover_days,avg_median_delivery_days,avg_median_promised_days
0,states at or above 12%,8,18792.0,2.25,18.38,31.88
1,states below 12%,19,77411.0,2.18,15.58,31.32


## 5. Q3 — do late orders get worse reviews?

Two cautions, both load-bearing:

1. **Coverage is not 100%.** Reviews are voluntary, so scores describe the
   customers who chose to review, not all customers.
2. **This is an association.** A hard route, a slow seller or a stock problem can
   produce both the delay and the low score. Nothing here isolates a causal
   effect, and no recommendation below is phrased as if it did.

In [14]:
q('''
SELECT
  count(*)                            AS n_sale_eligible,
  count(*) FILTER (WHERE has_review)  AS n_reviewed,
  round(100.0*count(*) FILTER (WHERE has_review)/count(*), 2) AS coverage_pct
FROM mart.fct_orders
WHERE is_sale_eligible AND in_analysis_window
''')

,n_sale_eligible,n_reviewed,coverage_pct
0,97910,97181,99.26


In [15]:
q("SELECT * FROM mart.kpi_reviews_by_punctuality")

,delivery_outcome,n_orders,mean_review_score,median_review_score,n_low_scores,pct_1_2_star,n_five_star,pct_5_star,pct_with_comment
0,late,6378,2.271,1.0,3980,62.40,1056,16.56,58.92
1,on time,89182,4.291,5.0,8248,9.25,55538,62.27,39.18


The two-bucket split is suggestive. The **monotone gradient** across degrees
of lateness is much stronger evidence, because a confounder would have to track
lateness by degree to reproduce it.

In [16]:
q("SELECT * FROM mart.kpi_reviews_by_delay_bucket")

,delay_bucket,n_orders,pct_of_reviewed_orders,mean_review_score,n_low_scores,pct_1_2_star,pct_with_comment
0,1. early by 8+ days,70673,73.96,4.318,6326,8.95,39.66
1,2. early by 1-7 days,17229,18.03,4.201,1763,10.23,37.24
2,3. on the promised day,1280,1.34,4.034,159,12.42,38.75
3,4. late 1-3 days,1851,1.94,3.291,595,32.14,48.35
4,5. late 4-7 days,1748,1.83,2.105,1182,67.62,59.73
5,6. late 8-15 days,1601,1.68,1.674,1281,80.01,64.71
6,7. late 16+ days,1178,1.23,1.727,922,78.27,66.47


Two details worth noticing:

- The jump is at the **promise boundary**, not at some absolute delivery speed:
  12.4% low scores on the promised day, 32.1% at one to three days late.
- Past roughly eight days the curve **plateaus** near 80%. Beyond a point,
  more lateness cannot lower a score that is already at the floor.
- Even perfectly on-time orders carry a ~9% low-score rate. That is a floor
  faster delivery cannot touch, and it caps what any logistics fix can achieve.

## 6. Q4 — what deserves attention, weighing rate against volume?

Ranking by rate alone picks tiny groups; ranking by volume alone picks whatever
is biggest. `excess_late_orders` is the arithmetic gap between a group's late
orders and what it would have at the national rate — an order count an
operations team can actually work through.

It is **not** a forecast of what a fix would save.

In [17]:
q('''
SELECT customer_state, n_orders, late_rate_pct, rate_gap_pp, excess_late_orders,
       n_reviewed_orders, mean_review_score
FROM mart.kpi_state_priority
ORDER BY excess_late_orders DESC LIMIT 8
''')

,customer_state,n_orders,late_rate_pct,rate_gap_pp,excess_late_orders,n_reviewed_orders,mean_review_score
0,RJ,12310,12.14,5.35,659.0,12172,3.965
1,BA,3253,12.17,5.38,175.0,3226,3.929
2,CE,1273,13.83,7.04,90.0,1267,3.940
3,ES,1992,10.74,3.95,79.0,1966,4.079
4,MA,713,17.53,10.74,77.0,709,3.829
5,AL,396,21.46,14.67,58.0,393,3.847
6,SC,3537,8.23,1.44,51.0,3510,4.134
7,PE,1587,9.64,2.85,45.0,1573,4.082


### The same data, ranked the naive way

Ranking by rate promotes states with a few hundred orders above RJ, which alone
accounts for the largest block of excess late orders in the country.

In [18]:
q('''
SELECT customer_state, n_orders, late_rate_pct, excess_late_orders
FROM mart.kpi_state_priority
ORDER BY late_rate_pct DESC LIMIT 8
''')

,customer_state,n_orders,late_rate_pct,excess_late_orders
0,AL,396,21.46,58.0
1,MA,713,17.53,77.0
2,SE,332,15.36,28.0
3,PI,475,13.89,34.0
4,CE,1273,13.83,90.0
5,RR,40,12.50,2.0
6,BA,3253,12.17,175.0
7,RJ,12310,12.14,659.0


### Multi-seller orders: why seller blame is withheld

1,278 orders were fulfilled by more than one seller. The review belongs to the
*order*, so attributing it to a single seller would be a fabricated number.
Seller-level analysis in this project is restricted to logistics.

In [19]:
q('''
SELECT
  CASE WHEN is_multi_seller THEN 'multi-seller' ELSE 'single-seller' END AS order_type,
  count(*)                        AS n_orders,
  count(*) FILTER (WHERE has_review) AS n_reviewed,
  round(avg(review_score), 3)     AS mean_review_score,
  round(100.0*count(*) FILTER (WHERE is_late)/count(*), 2) AS late_rate_pct
FROM mart.fct_orders
WHERE has_delivery_measurement AND in_analysis_window
GROUP BY 1 ORDER BY 1
''')

,order_type,n_orders,n_reviewed,mean_review_score,late_rate_pct
0,multi-seller,1272,1258,2.862,1.02
1,single-seller,94931,94302,4.173,6.87


## 7. What this notebook does not establish

- **No causality.** Every relationship here is an association.
- **No profit.** Item value, freight and payments are three measures; the dataset
  has no cost, tax, refund or commission data.
- **No full customer picture.** Reviews cover part of orders, and the dataset
  ends in 2018 — nothing here describes current performance.
- **Small groups stay small.** Several states and categories carry a few hundred
  orders; their rates move a lot on a handful of events, and `n` is printed
  beside every rate for exactly that reason.

In [20]:
con.close()
print("done")

done
